# 03 — Enable recurrent materialization and run batch inference

Adapted from the official [3.Enable-recurrent-materialization-run-batch-inference.ipynb](https://github.com/Azure/azureml-examples/blob/65eceac7ff5cd2417af5f24d38192ab8e445dd85/sdk/python/featurestore_sample/notebooks/sdk_only/3.Enable-recurrent-materialization-run-batch-inference.ipynb) notebook
(MIT). The official code is kept as-is except for the local-execution changes listed in the
[README](README.md#changes-from-the-official-notebooks). Talk track: Slides 11 and 19 in [FEATURE_STORE_TALK_TRACK.md](FEATURE_STORE_TALK_TRACK.md).

Schedule **recurrent materialization on serverless Spark**, then run the batch inference pipeline: the built-in feature retrieval component reads the offline store on serverless Spark, and the model scores the joined data on the AML compute cluster.

| Label | Where the work runs |
|---|---|
| **[Local]** | Your computer: the notebook kernel, local Spark, or local Python |
| **[Control plane]** | Azure ML or Azure Resource Manager API call made from your computer |
| **[Serverless Spark]** | Azure ML managed Spark job submitted from your computer |

In this tutorial series you will experience how features seamlessly integrates all the phases of ML lifecycle: Prototyping features, training and operationalizing.

In the previous part of the tutorial you learnt to experiment with features, train the model and register the model along with the feature-retrieval spec. In this tutorial you will learn how to run batch inference for the registered model.

You will perform the following:
- Enable recurrent materialization for the `transactions` feature set
- Run batch inference pipeline on the registered model

## Prerequisites

- Run the notebooks in order from this folder with the repository `.venv` kernel. Run
  [00_prepare_sample_data.ipynb](00_prepare_sample_data.ipynb) first.
- Sign in with `az login`. The notebooks use `AzureCliCredential` instead of `AzureMLOnBehalfOfCredential`,
  which works only inside Azure ML compute.
- Configure the feature store section of `workshop/.env`; see [README](README.md#configuration).
- Your account needs `Owner`, or `Contributor` plus `User Access Administrator`, on the resource group.

#### **[Local]** Load the local configuration
The official notebook runs in Azure ML studio, which provides the `AZUREML_ARM_*` environment variables. This cell provides the same values locally from `workshop/.env`. This notebook doesn't need Spark.

In [ ]:
# Local equivalent of the Azure ML studio Serverless Spark notebook session.
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

load_dotenv(Path.cwd().parents[1] / ".env", override=True)
os.environ["AZUREML_ARM_SUBSCRIPTION"] = os.environ["AZURE_SUBSCRIPTION_ID"]
os.environ["AZUREML_ARM_RESOURCEGROUP"] = os.environ["AZURE_RESOURCE_GROUP"]
os.environ["AZUREML_ARM_WORKSPACE_NAME"] = os.environ["AZUREML_WORKSPACE_NAME"]

# Local copy for local Spark development, and the ADLS Gen2 copy used by registered feature sets.
local_data_root = (Path.cwd().parents[1] / "outputs" / "feature_store_sample_data" / "feature-store-prp").as_uri()
cloud_data_root = (
    f"abfss://{os.environ['FEATURE_STORE_SOURCE_FILESYSTEM']}"
    f"@{os.environ['FEATURE_STORE_SOURCE_STORAGE_ACCOUNT']}.dfs.core.windows.net/feature-store-prp"
)
print("start spark session" if "spark" in globals() else "local configuration loaded")

#### Setup root directory for the samples

In [ ]:
import os

# Local folder with the official featurestore_sample assets (featurestore/ and project/).
root_dir = os.path.join(os.getcwd(), "featurestore_sample")

if os.path.isdir(root_dir):
    print("The folder exists.")
else:
    print("The folder does not exist. Please create or fix the path")

#### Initialize the project workspace CRUD client
This is the current workspace where you will be running the tutorial notebook from

In [ ]:
### Initialize the MLClient of this project workspace
import os
from azure.ai.ml import MLClient
from azure.identity import AzureCliCredential

project_ws_sub_id = os.environ["AZUREML_ARM_SUBSCRIPTION"]
project_ws_rg = os.environ["AZUREML_ARM_RESOURCEGROUP"]
project_ws_name = os.environ["AZUREML_ARM_WORKSPACE_NAME"]

# connect to the project workspace
ws_client = MLClient(
    AzureCliCredential(process_timeout=60), project_ws_sub_id, project_ws_rg, project_ws_name
)

#### Initialize the feature store CRUD client
Ensure you update the `featurestore_name` to reflect what you created in part 1 of this tutorial

In [ ]:
from azure.ai.ml import MLClient
from azure.identity import AzureCliCredential

# feature store
featurestore_name = (
    os.environ["AZUREML_FEATURE_STORE_NAME"]  # use the same name from part #1 of the tutorial
)
featurestore_subscription_id = os.environ["AZUREML_ARM_SUBSCRIPTION"]
featurestore_resource_group_name = os.environ["AZUREML_ARM_RESOURCEGROUP"]

# feature store ml client
fs_client = MLClient(
    AzureCliCredential(process_timeout=60),
    featurestore_subscription_id,
    featurestore_resource_group_name,
    featurestore_name,
)

#### Initialize the feature store core sdk client

In [ ]:
# feature store client
from azureml.featurestore import FeatureStoreClient
from azure.identity import AzureCliCredential

featurestore = FeatureStoreClient(
    credential=AzureCliCredential(process_timeout=60),
    subscription_id=featurestore_subscription_id,
    resource_group_name=featurestore_resource_group_name,
    name=featurestore_name,
)

## Step 1: **[Control plane]** Enable recurrent materialization on the `transactions` featureset

In part 2 of this tutorial you enabled materialization and performed backfill on the transactions feature set. Backfill is an ondemand one-time operation to compute and store feature values in the materialization store. However when you want to perform inference of the model in production, you might want to keep the materilization store upto date by setting up recurrent materialization jobs. These jobs run on user defined schedule
The recurrent job schedule works in the following way: 
- A window is defined by the interval and frequency. E.g., interval = 3 and frequency = Hour define a 3-hour window
- The first window starts at the start_time defined in the RecurenceTrigger, and so on.
- The first recurrent job will be submitted at the begining of the next window after the update time.
- Later recurrent jobs will be submitted at every window after the first job.

As explained in the previous parts of the tutorials, once data is materialized (backfill/recurrent materialization), feature retrieval will use the materialized data by default.

In [ ]:
from datetime import datetime
from azure.ai.ml.entities import RecurrenceTrigger

transactions_fset_config = fs_client.feature_sets.get(name="transactions", version="1")

# create a schedule that runs the materialization job every 3 hours
transactions_fset_config.materialization_settings.schedule = RecurrenceTrigger(
    # The service rejects a datetime start time (the SDK sends "2023-04-15 00:04:10").
    # Pass the ISO 8601 string that the official feature set YAML uses.
    interval=3, frequency="Hour", start_time="2023-04-15T00:04:10", time_zone="UTC"
)

fs_poller = fs_client.feature_sets.begin_create_or_update(transactions_fset_config)

# print(fs_poller.result()) fails: the service returns the start time as "04/15/2023 00:04:10 +00:00",
# which the SDK can't render as YAML. Print the saved schedule instead.
schedule = fs_poller.result().materialization_settings.schedule
print(f"Every {schedule.interval} {schedule.frequency}(s) from {schedule.start_time} ({schedule.time_zone})")

#### (Optional) Save the feature set asset  yaml with the updated settings

In [ ]:
## uncomment and run
# transactions_fset_config.dump(root_dir + "/featurestore/featuresets/transactions/featureset_asset_offline_enabled_with_schedule.yaml")

#### Track status of the recurrent materialization jobs in the feature store studio UI
This job will every three hours. 

__Action__:

1. Feel free to execute the next step for now (batch inference).
1. In three hours check the recurrent job status via the UI

## Step 2: **[Serverless Spark]** Run the batch-inference pipeline

In this step you will manually trigger the batch inference pipeline. In a production scenario, this could be trigerred by a ci/cd pipeline based on model registration/approval.

The batch-inference has the following steps:

1. Feature retrieval step: This use the same built-in feature retrieval component that we used in the training pipeline in the part 3 of the tutorial. Incase of training pipeline, we provided feature retreival spec as an input to the component. However in case of batch inference we will pass the registered model as the input and the component will look for feature retrieval spec in the model artifact. Another difference is that in case of training, the observation data had the target variable, however incase of batch inference it will not be present. The feature retrieval step will join the observation data with the features and output the data for batch inference.
1. Batch inference: This step uses the batch inference input data from previous step, runs inference on the model and outputs the data by appending the predicted value.

__Note:__ In this example we use a job for batch inference. You can also use Azure ML's batch endpoints.

In [ ]:
from azure.ai.ml import load_job  # will be used later

# set the batch inference  pipeline path
batch_inference_pipeline_path = (
    root_dir + "/project/fraud_model/pipelines/batch_inference_pipeline.yaml"
)
batch_inference_pipeline_definition = load_job(source=batch_inference_pipeline_path)

# run the training pipeline
batch_inference_pipeline_job = ws_client.jobs.create_or_update(
    batch_inference_pipeline_definition
)

# stream the run logs
ws_client.jobs.stream(batch_inference_pipeline_job.name)

#### Inspect the batch inference output data
1. In the pipeline view, double click on `inference_step` -> in `outputs` card, copy the `Data` field. It will be something like `azureml_995abbc2-3171-461e-8214-c3c5d17ede83_output_data_data_with_prediction:1`. 
1. Paste it in the below cell with name and version separately (notice that the last character is the version, separated by a `:`).
1. You will see the `predict_is_fraud` column generated by the batch inference pipeline

Explanation: Since we did not provide a `name` and `version` in the `outputs` of the `inference_step` in the batch inference pipeline (`/project/fraud_mode/pipelines/batch_inference_pipeline.yaml`), the system created an untracked data asset with a guid as name and version as 1. In the next cell we will be getting the data path from the asset and displaying it.

In [ ]:
import pandas as pd

# The official cell reads this output data asset with Spark. Locally, download it and read it with pandas.
inference_step = next(
    job for job in ws_client.jobs.list(parent_job_name=batch_inference_pipeline_job.name)
    if job.display_name == "inference_step"
)
inf_data_output = ws_client.data.get(
    name=f"azureml_{inference_step.name}_output_data_data_with_prediction",
    version="1",
)
print(inf_data_output.path)

download_path = os.path.join(os.getcwd(), "..", "..", "outputs", "feature_store_batch_inference")
ws_client.jobs.download(inference_step.name, output_name="data_with_prediction", download_path=download_path)
inf_output_df = pd.read_parquet(os.path.join(download_path, "named-outputs", "data_with_prediction"))
display(inf_output_df.head(5))
inf_output_df["predict_is_fraud"].value_counts()

## Cleanup

The recurrent schedule keeps running every three hours. Disable it after the demo; see [README](README.md#cleanup).

## Next steps
* [04 — Enable online materialization and run online inference](04_online_store_and_online_inference.ipynb)